In [ ]:
from src.partition_tests import run_partition_tests
from src.metamorphic_tests import run_metamorphic_tests
from src.helpers import *

# Load Data and Models

In [ ]:
DATA_PATH = "data/investigation_train_large_checked.csv"
ADDITIONAL_DATA_PATH = "data/synth_data_for_training.csv"

M1_ONNX_PATH  = "models/m1.onnx"
M2_ONNX_PATH   = "models/m2.onnx"

In [ ]:
df = pd.read_csv(DATA_PATH)
df_additional = pd.read_csv(ADDITIONAL_DATA_PATH)

leakage_cols = ["checked", "Ja", "Nee"]

y = df["checked"]
y_additional = df_additional["checked"]
X = df.drop(columns=leakage_cols, errors="ignore").fillna(0).astype(np.float32)
X_additional = df_additional.drop(columns=leakage_cols, errors="ignore").fillna(0).astype(np.float32)

print("X shape:", X.shape)
print("X_additional shape:", X_additional.shape)

X shape: (130000, 315)
X_additional shape: (12645, 315)


In [ ]:
m1_sess, m1_in   = load_onnx_session(M1_ONNX_PATH)
m2_sess, m2_in     = load_onnx_session(M2_ONNX_PATH)

m1_model  = make_onnx_model(m1_sess, m1_in)
m2_model   = make_onnx_model(m2_sess, m2_in)

# Performance Evaluation

## Model 1

In [ ]:
evaluate_onnx(m1_model, X, y, name="Model 1 (Original Data)")
evaluate_onnx(m1_model, X, y, name="Model 1 (Additional Data)")


===== Evaluation: Model 1 (Original Data) =====
Accuracy : 0.9000769230769231
Precision: 0.8003504241977131
Recall   : 0.4449856439704676
F1-score : 0.5719652036378015
ROC AUC  : 0.9243354731069295

===== Evaluation: Model 1 (Additional Data) =====
Accuracy : 0.9000769230769231
Precision: 0.8003504241977131
Recall   : 0.4449856439704676
F1-score : 0.5719652036378015
ROC AUC  : 0.9243354731069295


## Model 2

In [ ]:
evaluate_onnx(m2_model, X, y, name="Model 2 (Original Data)")
evaluate_onnx(m2_model, X, y, name="Model 2 (Additional Data)")


===== Evaluation: Model 2 (Original Data) =====
Accuracy : 0.8685
Precision: 0.7719575524949198
Recall   : 0.17529737489745692
F1-score : 0.2857142857142857
ROC AUC  : 0.8514942553497904

===== Evaluation: Model 2 (Additional Data) =====
Accuracy : 0.8685
Precision: 0.7719575524949198
Recall   : 0.17529737489745692
F1-score : 0.2857142857142857
ROC AUC  : 0.8514942553497904


# Run Partition and Metamorphic Tests

## Model 1 Original Data

In [ ]:
run_partition_tests(m1_model, X)


=== Language requirement met ===
Language requirement met: n=71680, mean risk=0.116
Language requirement not met: n=52185, mean risk=0.189
  [FAIL] Language requirement compliance has big impact on predicted risk.

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=34283, mean risk=0.151
Highest 20% language-related score: n=42352, mean risk=0.151
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=27316, mean risk=0.132
At least one attitude/motivation judgement recorded: n=102684, mean risk=0.155
  [FAIL] Attitude/motivation judgements by caseworkers have big impact on predicted risk.

=== Subjective communication judgement ===
Has communication judgement recorded: n=68601, mean risk=0.161
No communication judgement recorded: n=61399, mean risk=0.138
  [FAIL] Communication judgement by caseworkers has big impact on predicted risk.

=== Subjective appearance/presentation judgements ===
No appearance/pre

In [ ]:
run_metamorphic_tests(m1_model, X)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.1501
Mean score (flipped)  :       0.1510
Mean shift            :       +0.0009
Max |shift|           :       0.0968
Proportion |shift|>0.020:     0.240
  [FAIL] Metamorphic test for Language requirement met (flip 0/1) shows strong sensitivity.

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.1501
Mean score (flipped)  :       0.1501
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.1501
Mean score (flipped)  :       0.1501
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Attitude/motivation: baseline -> NONE recorded ===
Mean sc

## Model 1 Additional Data

In [ ]:
run_partition_tests(m1_model, X_additional)


=== Language requirement met ===
Language requirement met: n=7011, mean risk=0.114
Language requirement not met: n=5043, mean risk=0.189
  [FAIL] Language requirement compliance has big impact on predicted risk.

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=3350, mean risk=0.154
Highest 20% language-related score: n=4086, mean risk=0.147
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=2651, mean risk=0.130
At least one attitude/motivation judgement recorded: n=9994, mean risk=0.154
  [FAIL] Attitude/motivation judgements by caseworkers have big impact on predicted risk.

=== Subjective communication judgement ===
Has communication judgement recorded: n=6643, mean risk=0.162
No communication judgement recorded: n=6002, mean risk=0.134
  [FAIL] Communication judgement by caseworkers has big impact on predicted risk.

=== Subjective appearance/presentation judgements ===
No appearance/presentation

In [ ]:
run_metamorphic_tests(m1_model, X_additional)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.1487
Mean score (flipped)  :       0.1498
Mean shift            :       +0.0011
Max |shift|           :       0.0937
Proportion |shift|>0.020:     0.241
  [FAIL] Metamorphic test for Language requirement met (flip 0/1) shows strong sensitivity.

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.1487
Mean score (flipped)  :       0.1487
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.1487
Mean score (flipped)  :       0.1487
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Attitude/motivation: baseline -> NONE recorded ===
Mean sc

## Model 2 Original Data

In [ ]:
run_partition_tests(m2_model, X)


=== Language requirement met ===
Language requirement met: n=71680, mean risk=0.144
Language requirement not met: n=52185, mean risk=0.157
  [OK] 

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=34283, mean risk=0.154
Highest 20% language-related score: n=42352, mean risk=0.148
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=27316, mean risk=0.130
At least one attitude/motivation judgement recorded: n=102684, mean risk=0.155
  [FAIL] Attitude/motivation judgements by caseworkers have big impact on predicted risk.

=== Subjective communication judgement ===
Has communication judgement recorded: n=68601, mean risk=0.161
No communication judgement recorded: n=61399, mean risk=0.138
  [FAIL] Communication judgement by caseworkers has big impact on predicted risk.

=== Subjective appearance/presentation judgements ===
No appearance/presentation judgement recorded: n=55300, mean risk=0.137
At least one

In [ ]:
run_metamorphic_tests(m2_model, X)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.1501
Mean score (flipped)  :       0.1501
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.1501
Mean score (flipped)  :       0.1649
Mean shift            :       +0.0148
Max |shift|           :       0.1652
Proportion |shift|>0.020:     0.262
  [FAIL] Metamorphic test for Aggregate Dutch language indicators: baseline -> ALL LOW shows strong sensitivity.

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.1501
Mean score (flipped)  :       0.1449
Mean shift            :       -0.0052
Max |shift|           :       0.1200
Proportion |shift|>0.020:     0.151
  [FAIL] Metamorphic test for Aggregate Dutch language indicators: baselin

## Model 2 Additional Data

In [ ]:
run_partition_tests(m2_model, X_additional)


=== Language requirement met ===
Language requirement met: n=7011, mean risk=0.146
Language requirement not met: n=5043, mean risk=0.157
  [OK] 

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=3350, mean risk=0.155
Highest 20% language-related score: n=4086, mean risk=0.148
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=2651, mean risk=0.133
At least one attitude/motivation judgement recorded: n=9994, mean risk=0.156
  [FAIL] Attitude/motivation judgements by caseworkers have big impact on predicted risk.

=== Subjective communication judgement ===
Has communication judgement recorded: n=6643, mean risk=0.162
No communication judgement recorded: n=6002, mean risk=0.139
  [FAIL] Communication judgement by caseworkers has big impact on predicted risk.

=== Subjective appearance/presentation judgements ===
No appearance/presentation judgement recorded: n=5341, mean risk=0.138
At least one appearanc

In [ ]:
run_metamorphic_tests(m2_model, X_additional)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.1516
Mean score (flipped)  :       0.1516
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.1516
Mean score (flipped)  :       0.1663
Mean shift            :       +0.0147
Max |shift|           :       0.1368
Proportion |shift|>0.020:     0.263
  [FAIL] Metamorphic test for Aggregate Dutch language indicators: baseline -> ALL LOW shows strong sensitivity.

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.1516
Mean score (flipped)  :       0.1463
Mean shift            :       -0.0053
Max |shift|           :       0.1096
Proportion |shift|>0.020:     0.153
  [FAIL] Metamorphic test for Aggregate Dutch language indicators: baselin